In [0]:
from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite

In [0]:
import re
import numpy as np
import pandas as pd
import os
import string
import json
from openpyxl.utils import get_column_letter, column_index_from_string
from fuzzywuzzy import process
from fuzzywuzzy import fuzz
from collections import defaultdict
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from functools import reduce
from operator import or_
from pyspark.sql.types import DateType, TimestampType

In [0]:
#Define target fields for mapping - WC Loss Run
wc_target_fields = ['Claim number', 'Claim type (IND/MO/RO)', 'Date of loss', 'Date claim reported to TPA','Date claim reported to Client / Employer','Claim status (Open/Closed)','Date closed (if closed claim)','Net Total Incurred','Net Total Paid','Net Medical Paid','Net Indemnity Paid','Net Expense Paid', 'Net Medical Incurred', 'Litigation indicator (Yes/No) ','Date of hire (employee hire date)','Date of birth or age (employee date of birth)','Benefit State / Jurisdiction State','Loss description','Cause Description','Body part ','Source of injury','Nature of Injury Desc','MO to IND Conversion Indicator (if included in the WC loss run)','MO to IND Conversion Date (if included in the WC loss run)','TTD Indicator (if included in the WC loss run)','LWDs due to TTD (if included in the WC loss run)','Occupation']

In [0]:
#Define target fields for mapping - LIAB Loss Run
#Order matches rows 21-61 of the 'Liability Loss Run - ALGLPL ' sheet in Consultant_Triage_Template_New.xlsx (hierarchy rows 62-66 are not mapped).
#Where a field matches an existing canonical name in the TPA mapping dictionaries, that name is reused so dictionary matches still apply;
#the remaining fields are mapped by the LLM pass unless aliases are added to the mapping dictionaries.
liab_target_fields = [
    'Claim number',                                   # Claim number
    'Occurrence indicator or number',                 # Occurrence indicator or number
    'Coverage (AL/GL/PL)',                            # Major coverage code (AL/GL)
    'Detailed Claim Type / Minor Coverage',           # Minor coverage code (comp/collision vs. 3rd party, etc)
    'Claim type (BI, PD, record only, etc.)',         # Claim type (BI, PD, record only, etc.)
    'Date of loss',                                   # Date of loss
    'Time of day',                                    # Time of day
    'Date claim reported to TPA',                     # Date claim reported to TPA
    'Date claim reported to Client / Employer',       # Date claim reported to client
    'Claim status (Open/Closed)',                     # Claim status (open/closed/re-open)
    'Date closed (if closed claim)',                  # Date closed (if closed claim)
    'Date re-opened (if re-opened claim)',            # Date re-opened (if re-opened claim)
    'Net Total Incurred',                             # Total incurred (net, preferred)
    'Net Total Paid',                                 # Net total paid
    'Net BI paid',                                    # Net BI paid
    'Net PD paid',                                    # Net PD paid
    'Net Expense Paid',                               # Net expense paid
    'Net BI incurred',                                # Net BI incurred
    'Net PD incurred',                                # Net PD incurred
    'Net expense incurred',                           # Net expense incurred
    'Type of vehicle (for AL)',                       # Type of vehicle (for AL)
    'Claimant date of birth or age',                  # Claimant date of birth or age
    'Insured driver date of birth or age',            # Insured driver date of birth or age
    'Benefit State / Jurisdiction State',             # Benefit state / jurisdiction state
    'Accident state',                                 # Accident state
    'Loss description',                               # Loss description (loss narrative)
    'Cause Description',                              # Cause / type description
    'Litigation indicator (Yes/No) ',                 # Litigation indicator (yes/no)
    'Legal representation indicator for claimant',    # Legal representation indicator for claimant
    'Date of plaintiff legal representation',         # Date of plaintiff legal representation
    'Plaintiff attorney / firm name',                 # Plaintiff attorney / firm name
    'Date suit filed',                                # Date suit filed
    'Date suit received (by insured)',                # Date suit received (by insured)
    'Date suit received (by insurer)',                # Date suit received (by insurer)
    'Defense attorney indicator',                     # Defense attorney indicator
    'Defense attorney / firm name',                   # Defense attorney / firm name
    'Date defense attorney assigned',                 # Date defense attorney assigned
    'Indicator for action over claims',               # Indicator for action over claims
    'Medicare / medicare set aside indicator',        # Medicare / medicare set aside indicator
    'Subrogation indicator',                          # Subrogation indicator
    'Tender indicator',                               # Tender indicator
]

In [0]:
#Define target fields for mapping - TTD
ttd_target_fields = ["Date of Loss","TTD Indicator","LWD's due to TTD","Claim type","Payment Type (if Calculated from Payment Report)"]

In [0]:
#Define target fields for mapping - WC PTR (rows 11-18 of 'WC Payment Transaction Report')
wc_ptr_target_fields = ['Claim Number','Date of loss','Description of Payment / Payment Type','Payment Amount / Transaction Amount','Payment Category (Indemnity / Medical / Expense)','Date of payment','Payee','Pay Code']
#Define target fields for mapping - LIAB PTR (rows 11-19 of 'Liability Payment Trans Report'). Canonical PTR names are reused so the PTR mapping dictionary still applies
liab_ptr_target_fields = ['Claim Number','Date of loss','Description of Payment / Payment Type','Payment Amount / Transaction Amount','Payment Category (Indemnity / Medical / Expense)','Date of payment','Payee','Payee Tax ID/FEIN number','Pay Code']

In [0]:
def get_full_path(file):
   return os.path.join(directory_path, file)

In [0]:
#Normalizes sheet names so lookups ignore case, spaces, underscores and hyphens (e.g. "WC Loss Run" matches "WC_Loss_Run", "ttd" matches "TTD")
def normalize_sheet_name(name) -> str:
   return re.sub(r"[^a-z0-9]", "", str(name).lower())

#Returns the actual sheet name in sheet_names that matches target (after normalization), or None if not present
def find_sheet(sheet_names, target):
   target_norm = normalize_sheet_name(target)
   for s in sheet_names:
       if normalize_sheet_name(s) == target_norm:
           return s
   return None

#Looks through paths, determines which project files exist (as independent files or as sheets within loss run files), and reads them into memory as pandas dataframes. Used for job-based loop process
def load_client_sources(
   wc_loss_run_path: str = "",
   ttd_path: str = "",
   ptr_path: str = "",
   liab_loss_run_path: str = "",
   wc_ptr_path: str = "",
   liab_ptr_path: str = ""
):
   """
   At least one loss run (WC and/or LIAB) is required; both may be provided.
   Loss runs:
       - WC_Loss_Run sheet:   from the WC workbook, otherwise from the LIAB workbook if it contains one
       - LIAB_Loss_Run sheet: from the LIAB workbook, otherwise from the WC workbook if it contains one
   TTD (optional), in priority order:
       1) standalone TTD file (sheet named TTD, or the first sheet if no sheet matches)
       2) TTD sheet within the WC loss run workbook, then the LIAB loss run workbook
   PTR (optional) - WC PTR and LIAB PTR are separate. WC PTR, in priority order:
       1) standalone WC_PTR file (wc_ptr_path)
       2) standalone unlabeled PTR file (ptr_path), only if the WC loss run is the only loss run present
       3) 'WC_PTR' or 'PTR' sheet within the WC loss run workbook
       4) 'WC_PTR' sheet within the LIAB loss run workbook
      LIAB PTR follows the same rules with WC and LIAB swapped.
      An unlabeled PTR file when both loss runs are present is ambiguous: it is ignored and a warning is printed.
   Returns:
       df_loss_run_wc    (None if no WC loss run found)
       df_loss_run_liab  (None if no LIAB loss run found)
       df_ttd            (None if not found)
       df_wc_ptr         (None if not found)
       df_liab_ptr       (None if not found)
   """
   if not wc_loss_run_path and not liab_loss_run_path:
       raise ValueError("At least one of wc_loss_run_path or liab_loss_run_path is required")

   # Read sheet names of each provided loss run workbook once (no data loaded)
   loss_run_books = {}  # label -> (path, sheet_names)
   for label, path in (("WC", wc_loss_run_path), ("LIAB", liab_loss_run_path)):
       if path:
           try:
               with pd.ExcelFile(path) as xl:
                   loss_run_books[label] = (path, xl.sheet_names)
           except Exception as e:
               raise ValueError(f"{label} loss run file was provided but could not be opened: {path}. Error: {e}")

   # Searches the loss run workbooks for the first matching (workbook label, sheet name) candidate and loads it
   def from_loss_runs(description, candidates, **read_kwargs):
       for label, sheet in candidates:
           if label in loss_run_books:
               path, sheet_names = loss_run_books[label]
               actual = find_sheet(sheet_names, sheet)
               if actual is not None:
                   df = pd.read_excel(path, sheet_name=actual, **read_kwargs)
                   print(f"Loaded {description} from sheet '{actual}' in {label} loss run workbook: {path}")
                   return df
       return None

   # Loads a standalone optional file (header row only - used for mapping)
   def from_standalone(description, sheet_candidates, path):
       try:
           with pd.ExcelFile(path) as xl:
               actual = next((s for s in (find_sheet(xl.sheet_names, c) for c in sheet_candidates) if s), xl.sheet_names[0])
           df = pd.read_excel(path, sheet_name=actual, nrows=1)
           print(f"Loaded {description} from standalone file (sheet '{actual}'): {path}")
           return df
       except Exception as e:
           raise ValueError(f"{description} file was provided but could not be read: {path}. Error: {e}")

   # WC Loss Run (full data - used for DQ checks)
   df_loss_run_wc = from_loss_runs("WC Loss Run", [("WC", "WC_Loss_Run"), ("LIAB", "WC_Loss_Run")])
   if wc_loss_run_path and df_loss_run_wc is None:
       raise ValueError(f"Could not find a 'WC_Loss_Run' sheet in WC loss run file: {wc_loss_run_path}")

   # LIAB Loss Run (header row only - used for mapping)
   df_loss_run_liab = from_loss_runs("LIAB Loss Run", [("LIAB", "LIAB_Loss_Run"), ("WC", "LIAB_Loss_Run")], nrows=1)
   if liab_loss_run_path and df_loss_run_liab is None:
       raise ValueError(f"Could not find a 'LIAB_Loss_Run' sheet in LIAB loss run file: {liab_loss_run_path}")

   has_wc, has_liab = df_loss_run_wc is not None, df_loss_run_liab is not None

   # TTD: standalone file first, then sheets within the WC workbook, then the LIAB workbook
   if ttd_path:
       df_ttd = from_standalone("TTD", ["TTD", "WC_TTD"], ttd_path)
   else:
       df_ttd = from_loss_runs("TTD", [("WC", "TTD"), ("WC", "WC_TTD"), ("LIAB", "TTD"), ("LIAB", "WC_TTD")], nrows=1)

   # PTR: decide which line an unlabeled standalone PTR file belongs to
   generic_ptr_line = None
   if ptr_path:
       if has_wc and has_liab:
           print(f"WARNING: PTR file '{ptr_path}' is not labeled WC_PTR or LIAB_PTR and both WC and LIAB loss runs are present, "
                 f"so it cannot be assigned and will be ignored. Rename it to <client>_WC_PTR.xlsx or <client>_LIAB_PTR.xlsx.")
       else:
           generic_ptr_line = "WC" if has_wc else "LIAB"
           labeled = wc_ptr_path if has_wc else liab_ptr_path
           if labeled:
               print(f"WARNING: both '{labeled}' and unlabeled '{ptr_path}' were provided for {generic_ptr_line} PTR; using the labeled file.")
               generic_ptr_line = None

   # WC PTR
   wc_ptr_file = wc_ptr_path or (ptr_path if generic_ptr_line == "WC" else "")
   if wc_ptr_file:
       df_wc_ptr = from_standalone("WC PTR", ["WC_PTR", "PTR"], wc_ptr_file)
   else:
       df_wc_ptr = from_loss_runs("WC PTR", [("WC", "WC_PTR"), ("WC", "PTR"), ("LIAB", "WC_PTR")], nrows=1)

   # LIAB PTR
   liab_ptr_file = liab_ptr_path or (ptr_path if generic_ptr_line == "LIAB" else "")
   if liab_ptr_file:
       df_liab_ptr = from_standalone("LIAB PTR", ["LIAB_PTR", "PTR"], liab_ptr_file)
   else:
       df_liab_ptr = from_loss_runs("LIAB PTR", [("LIAB", "LIAB_PTR"), ("LIAB", "PTR"), ("WC", "LIAB_PTR")], nrows=1)

   if not has_wc:
       print("WC Loss Run data not found - WC mapping and DQ assessment will be skipped.")
   if not has_liab:
       print("LIAB Loss Run data not found in standalone file or loss run workbook sheets.")
   if df_ttd is None:
       print("TTD data not found in standalone file or loss run workbook sheets.")
   if df_wc_ptr is None:
       print("WC PTR data not found in standalone files or loss run workbook sheets.")
   if df_liab_ptr is None:
       print("LIAB PTR data not found in standalone files or loss run workbook sheets.")
   return df_loss_run_wc, df_loss_run_liab, df_ttd, df_wc_ptr, df_liab_ptr

In [0]:
#Operates similarly to above function, but used for manual non-looped process runs. Finds project files by name in a directory, then delegates loading to load_client_sources
def find_files(directory_path):
    # Order matters: labeled PTR suffixes must be checked before the generic _PTR suffix
    suffixes = [
        ('_WC_Loss_Run.xlsx',   'wc_loss_run_path'),
        ('_LIAB_Loss_Run.xlsx', 'liab_loss_run_path'),
        ('_WC_PTR.xlsx',        'wc_ptr_path'),
        ('_LIAB_PTR.xlsx',      'liab_ptr_path'),
        ('_TTD.xlsx',           'ttd_path'),
        ('_PTR.xlsx',           'ptr_path'),
    ]
    paths = {key: "" for _, key in suffixes}
    for file in os.listdir(directory_path):
        for suffix, key in suffixes:
            if file.endswith(suffix):
                paths[key] = os.path.join(directory_path, file)
                break
    if not paths['wc_loss_run_path'] and not paths['liab_loss_run_path']:
        raise ValueError(f"No '_WC_Loss_Run.xlsx' or '_LIAB_Loss_Run.xlsx' file found in {directory_path}")
    client_string = extract_substring_from_string(paths['wc_loss_run_path'] or paths['liab_loss_run_path'])
    df_loss_run_wc, df_loss_run_liab, df_ttd, df_wc_ptr, df_liab_ptr = load_client_sources(**paths)
    return df_loss_run_wc, df_loss_run_liab, df_ttd, df_wc_ptr, df_liab_ptr, client_string

In [0]:
#Reads mapping dictionary .json files and returns a dictionary
def read_json_to_dict(file_path):
    try:
        with open(file_path, 'r') as file:
            data = json.load(file)
        return data
    except FileNotFoundError:
        print(f"Error: File not found: {file_path}")
        return None
    except json.JSONDecodeError:
        print(f"Error: Invalid JSON format in: {file_path}")
        return None

In [0]:
def extract_substring_from_string(input_string):
    # Use a regular expression to extract the desired substring
    match = re.search(r'.*/([^_]+)_.*', input_string)
    if match:
        return match.group(1)
    else:
        return None

In [0]:
#Converts index number to Excel Column Letter format
def index_to_excel_col(n):
   result = ""
   while n >= 0:
       result = chr(n % 26 + ord('A')) + result
       n = n // 26 - 1
   return result

In [0]:
#Uses fuzzy matching technique with threshold >90 to renamed columns based on mapping dictionary
def column_matcher(df, mapping_dict):
    df_clone = df.copy()
    # Step 1: Gather all alias keys and define threshold
    all_aliases = list(mapping_dict.keys())
    threshold = 90
    all_matches = []
    # Step 2: Match each column to the best alias (if above threshold)
    for col in df_clone.columns:
        match, score = process.extractOne(col, all_aliases)
        if score >= threshold:
            canonical = mapping_dict[match]
            all_matches.append((col, canonical, score))
    # Step 3: Resolve conflicts — only best column per canonical target
    conflict_tracker = defaultdict(list)
    for col, canonical, score in all_matches:
        conflict_tracker[canonical].append((col, score))
    # Track best matches
    best_matches = {}  # {original_col: canonical_name}
    assigned_columns = set()
    for canonical, candidates in conflict_tracker.items():
        best_col, best_score = sorted(candidates, key=lambda x: x[1], reverse=True)[0]
        best_matches[best_col] = canonical
        assigned_columns.add(best_col)
    # Step 4: Build the renaming map
    rename_map = {}
    for col in df_clone.columns:
        if col in best_matches:
            # Best match — rename to canonical name
            rename_map[col] = best_matches[col]
        else:
            # Not best match or no match — mark as UNMAPPED
            rename_map[col] = col
    # Step 5: Apply the renaming
    df_renamed = df_clone.rename(columns=rename_map)
    return df_renamed



In [0]:
#For unmapped columns after mapping dictionary pass, prompts LLM to map remaining columns to target schema
def llm_check(remaining_targets, unmapped_source):
   """
   remaining_targets: list of target schema fields
   unmapped_source: list of available column names (not a DataFrame)
   """
   from pyspark.sql import SparkSession
   spark = SparkSession.builder.getOrCreate()
   # Create target field dataframe
   targets_df = spark.createDataFrame([(col,) for col in remaining_targets], ['target_field'])
   targets_df.createOrReplaceTempView("target_fields_to_map")
   # Prepare input list as comma-separated string with escaped single quotes
   input_list_str = ", ".join([f"""`{c.replace("'", "''")}`""" for c in unmapped_source])
   # Build SQL query string
   sql_prompt = f"""
   CREATE OR REPLACE TEMP VIEW llm_column_mapping AS
   SELECT
     target_field,
     ai_query(
       'databricks-gpt-oss-20b',
       CONCAT(
         'You are mapping raw Excel column names to a target schema field. ',
         'Given the available input columns: {input_list_str}. ',
         'Choose the input column that best matches the target field: ', target_field, '. ',
         'If none are a reasonable match, respond exactly with "UNMAPPED".'
       )
     ) AS matched_column
   FROM target_fields_to_map
   """
   spark.sql(sql_prompt)
   return spark.table("llm_column_mapping").toPandas()

In [0]:
#Integrates mapping dictionary process and LLM mapping process to fully map incoming columns
def triage_mapper(df, target_fields, mapping_dict):
    # Step 1: Read Excel
    columns = df.columns.tolist()
    col_to_letter = {
        col: index_to_excel_col(idx)
        for idx, col in enumerate(columns)
    }
    # Step 2: Match Excel columns to aliases
    threshold = 90
    all_aliases = list(mapping_dict.keys())
    # Build match list: (excel_col, excel_index, matched_canonical, score)
    match_candidates = []
    column_to_target = {}
    for idx, col in enumerate(columns):
        match, score = process.extractOne(col, mapping_dict.keys())
        if match and score >= threshold:
            canonical_field = mapping_dict[match]
            match_candidates.append((canonical_field, col, idx, score))
        else:
            column_to_target[col] = 'UNMAPPED'
    # Step 3: Pick best match per target field
    best_matches = {}
    seen_indices = set()
    canonical_to_col = {}
    for field in target_fields:
        candidates = [(col, idx, score) for (canon, col, idx, score) in match_candidates if canon == field]
        if candidates:
            # Choose the highest scoring match for this field
            sorted_candidates = sorted(candidates, key=lambda x: x[2], reverse=True)
            for col, idx, score in sorted_candidates:
                if idx not in seen_indices:
                    best_matches[field] = idx
                    canonical_to_col[field] = col
                    column_to_target[col] = field
                    seen_indices.add(idx)
                    break
                else:
                    best_matches[field] = None
                    canonical_to_col[field] = None
        else:
            best_matches[field] = None
            canonical_to_col[field] = None
    # Step 4: Build result DataFrame
    for col in columns:
        if col not in column_to_target:
            column_to_target[col] = 'UNMAPPED'
    matched_column_to_target = {k: v for k, v in column_to_target.items() if v != 'UNMAPPED'}
    target_to_column = {v: k for k, v in matched_column_to_target.items()}
    unmapped_source = [col for col, mapped in column_to_target.items() if mapped == 'UNMAPPED']
    mapped_targets = set(val for val in column_to_target.values() if val != 'UNMAPPED')
    remaining_targets = [t for t in target_fields if t not in mapped_targets]
    #Step 5: Use LLM to make second pass on unmatched fields
    if not remaining_targets:
        llm_results = []
        print("No remaining targets for LLM Mapping")
    else:
        llm_results = llm_check(remaining_targets, unmapped_source)
    #Step 6: Build Final Output
    final_output = []
    # Combine all target fields
    #all_target_fields = list(set(llm_results['target_field'].tolist()) | set(target_to_column.keys()))
    all_target_fields = target_fields
    for target in all_target_fields:
        if target in target_to_column:
            input_col = target_to_column[target]
            letter = col_to_letter.get(input_col, 'UNMAPPED')
            source = 'DICT'
        else:
    # Look up LLM result
            match_row = llm_results[llm_results['target_field'] == target]
            if not match_row.empty:
                input_col = match_row.iloc[0]['matched_column']
                if input_col == 'UNMAPPED':
                    letter = 'UNMAPPED'
                    source = 'UNMAPPED'
                else:
                    letter = col_to_letter.get(input_col, 'UNMAPPED')
                    source = 'LLM'
            else:
                # Not mapped at all
                letter = 'UNMAPPED'
                source = 'UNMAPPED'
        final_output.append((target, letter, source))
    # Create DataFrame
    final_mapping_df = pd.DataFrame(final_output, columns=['Field Name', 'Matched Column Letter', 'Source'])
    return final_mapping_df


In [0]:
#Writes a triage_mapper result into a Consultant Triage sheet: matched column letters into letter_col and mapping source (DICT/LLM/UNMAPPED) into source_col,
#starting at start_row. Warns if a row being written has no field label in column B (i.e. the template layout no longer matches the target field list)
def write_mapping_to_sheet(ws, mapping_df, start_row, letter_col='C', source_col='D'):
   for offset, (letter, source) in enumerate(zip(mapping_df['Matched Column Letter'], mapping_df['Source'])):
       row = start_row + offset
       if ws[f"B{row}"].value in (None, ""):
           print(f"WARNING: '{ws.title}' row {row} has no field label in column B - check the template layout / target field list")
       ws[f"{letter_col}{row}"] = letter
       ws[f"{source_col}{row}"] = source

#openpyxl drops Excel 'x14' data validations (drop-downs whose list lives on another sheet, e.g. Industry Group, Milestone, Yes/No) when it loads a workbook,
#so they would be missing from the saved output. This re-reads them from the template file and re-adds them to wb as standard data validations.
def restore_x14_data_validations(template_path, wb):
   import zipfile
   import posixpath
   import xml.etree.ElementTree as ET
   from openpyxl.worksheet.datavalidation import DataValidation
   ns = {
       "main": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
       "rel":  "http://schemas.openxmlformats.org/package/2006/relationships",
       "x14":  "http://schemas.microsoft.com/office/spreadsheetml/2009/9/main",
       "xm":   "http://schemas.microsoft.com/office/excel/2006/main",
   }
   r_id = "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"
   bool_attrs = {"allowBlank": "allow_blank", "showInputMessage": "showInputMessage", "showErrorMessage": "showErrorMessage", "showDropDown": "showDropDown"}
   text_attrs = ["promptTitle", "prompt", "errorTitle", "error", "errorStyle", "operator"]
   restored = 0
   with zipfile.ZipFile(template_path) as z:
       workbook_xml = ET.fromstring(z.read("xl/workbook.xml"))
       rels_xml = ET.fromstring(z.read("xl/_rels/workbook.xml.rels"))
       rel_targets = {r.get("Id"): r.get("Target") for r in rels_xml.findall("rel:Relationship", ns)}
       for sheet in workbook_xml.find("main:sheets", ns):
           name, target = sheet.get("name"), rel_targets.get(sheet.get(r_id), "")
           sheet_path = target.lstrip("/") if target.startswith("/") else posixpath.normpath(posixpath.join("xl", target))
           if name not in wb.sheetnames or sheet_path not in z.namelist():
               continue
           for dv in ET.fromstring(z.read(sheet_path)).iter(f"{{{ns['x14']}}}dataValidation"):
               sqref = dv.find("xm:sqref", ns)
               if sqref is None or not sqref.text:
                   continue
               f1 = dv.find("x14:formula1/xm:f", ns)
               f2 = dv.find("x14:formula2/xm:f", ns)
               kwargs = {py: dv.get(x) in ("1", "true") for x, py in bool_attrs.items() if dv.get(x) is not None}
               kwargs.update({a: dv.get(a) for a in text_attrs if dv.get(a) is not None})
               wb[name].add_data_validation(DataValidation(
                   type=dv.get("type"),
                   formula1=f1.text if f1 is not None else None,
                   formula2=f2.text if f2 is not None else None,
                   sqref=sqref.text,
                   **kwargs
               ))
               restored += 1
   return restored

In [0]:
def hierarchy_profiler(df, hierarchy_fields, metric_field):
    # Container for profiling results
    profiling_results = []
    # Loop through optional fields and build summaries
    for field in hierarchy_fields:
        if field in df.columns:
            summary_df = (
                df.groupBy(field)
                    .agg(
                        F.countDistinct("Claim number").alias("Claim_Count"),
                        F.sum(F.col(metric_field)).alias("Total_Incurred_Sum")
                    )
                    .withColumnRenamed(field, "Field_Value")
                    .withColumn("Field_Name", F.lit(field))
                .select("Field_Name", "Field_Value", "Claim_Count", "Total_Incurred_Sum")
            )
            profiling_results.append(summary_df)
        else:
            # Optional: create a placeholder row if field is missing
            missing_df = df.sql_ctx.createDataFrame(
                [(field, "FIELD_NOT_MAPPED", 0, 0.0)],
                ["Field_Name", "Field_Value", "Claim_Count", "Total_Incurred_Sum"]
            )
            profiling_results.append(missing_df)
    # Combine all field summaries into one table
    if profiling_results:
        profiling_output = profiling_results[0]
        for s in profiling_results[1:]:
            profiling_output = profiling_output.unionByName(s)
    else:
        profiling_output = df.sql_ctx.createDataFrame([], schema="Field_Name STRING, Field_Value STRING, Claim_Count BIGINT, Total_Incurred_Sum DOUBLE") 
    return profiling_output

In [0]:
def check_column_status(df: DataFrame, columns: list[str]) -> dict:
   """
   Checks each requested column for presence or duplicates (case-insensitive).
   Returns a dict: {column_name: status_message or None if OK}.
   """
   lower_cols = [c.lower() for c in df.columns]
   col_status = {}
   for c in columns:
       matching = [col for col in df.columns if col.lower() == c.lower()]
       if len(matching) == 0:
           col_status[c] = "UNMAPPED FIELD"
       elif len(matching) > 1:
           col_status[c] = f"DUPLICATE FIELD: {c}"
       else:
           col_status[c] = None  # OK
   return col_status

In [0]:
BAD_DATE_STRINGS = {"00/00/0000", "0/0/0000", "00-00-0000", "0000-00-00", ""}
date_cols = ["Date of loss", "Date claim reported to TPA", "Date claim reported to Client / Employer", "Date closed (if closed claim)","Date of birth or age (employee date of birth)", "Date of hire (employee hire date)","MO to IND Conversion Date (if included in the WC loss run)"]

def normalize_and_parse_dates(df, target_date_cols):
   present = [c for c in target_date_cols if c in df.columns]
   for c in present:
       dtype = df.schema[c].dataType
       # If it's already date/timestamp, just normalize to date (no format parsing)
       if isinstance(dtype, (DateType, TimestampType)):
           df = df.withColumn(c, F.to_date(F.col(c)))
           continue
       # Otherwise treat as string with flexible parsing
       s = F.trim(F.col(c).cast("string"))
       parsed = F.coalesce(
           F.to_date(s, "MM/dd/yyyy"),
           F.to_date(s, "M/d/yyyy"),
           F.to_date(s, "yyyy-MM-dd"),
           F.to_date(s, "yyyy-MM-dd HH:mm:ss"),
       )
       df = df.withColumn(
           c,
           F.when(s.isin(*BAD_DATE_STRINGS), F.lit(None)).otherwise(parsed)
       )
   return df

In [0]:
def run_dq_checks(df: DataFrame) -> DataFrame:
   """
   Applies data quality checks and returns the entire dataframe with DQ check result columns.
   If a check's required column(s) are missing, that check column will contain "UNMAPPED FIELD".
   Adds:
     - DQ_Status: FAIL if any check failed, otherwise PASS
     - DQ_Mapping_Status: UNMAPPED if any check references unmapped fields, otherwise MAPPED
   """
   # Date Column Normalization
   df = normalize_and_parse_dates(df, date_cols)
   # Mandatory Fields Check
   dq = df
   mandatory_fields = ["Claim number", "Date of loss", "Date of birth or age (employee date of birth)", "Date of hire (employee hire date)", "Claim type (IND/MO/RO)", "Cause Description"]
   status = check_column_status(dq, mandatory_fields)
   issue = next((status[c] for c in mandatory_fields if status[c] is not None), None)
   if issue is not None:
       dq = dq.withColumn("Check_MandatoryFields_NotNull", F.lit(issue))
   else:
       # Fail if any mandatory field is null OR blank (after trimming as string)
       null_or_blank_exprs = []
       for c in mandatory_fields:
           col = F.col(c)
           null_or_blank_exprs.append(
               col.isNull() |
               (F.when(col.cast("string").isNotNull(), F.trim(col.cast("string")) == "").otherwise(F.lit(False)))
           )
       any_null_or_blank = reduce(or_, null_or_blank_exprs) if null_or_blank_exprs else F.lit(False)
       dq = dq.withColumn(
           "Check_MandatoryFields_NotNull",
           F.when(any_null_or_blank, "FAIL").otherwise("PASS")
       )
   # Date of loss not older than 36 months
   col_status = check_column_status(dq, ["Date of loss"])
   max_age_months = 36
   if col_status["Date of loss"]:
       dq1 = dq.withColumn("Check_DateOfLoss_TooOld", F.lit(col_status["Date of loss"]))
   else:
       max_date_loss = dq.agg(F.max("Date of loss").alias("MaxDateOfLoss")).collect()[0]["MaxDateOfLoss"]
       dq1 = (
           dq.withColumn(
               "Check_DateOfLoss_TooOld",
               F.when(
                   (F.col("Date of loss").isNull()) |
                   (F.months_between(F.lit(max_date_loss), F.col("Date of loss")) > max_age_months),
                   "FAIL"
               ).otherwise("PASS")
           )
       )
   # Record Only → Financial columns must be null
   financial_cols = ["Net Total Incurred", "Net Total Paid", "Net Medical Paid", "Net Indemnity Paid", "Net Expense Paid", "Net Medical Incurred"]
   required_for_dq2 = ["Claim type (IND/MO/RO)"] + financial_cols
   col_status = check_column_status(dq1, required_for_dq2)
   if any(col_status[c] for c in required_for_dq2):
       first_issue = next((col_status[c] for c in required_for_dq2 if col_status[c]), None)
       dq2 = dq1.withColumn("Check_RecordOnly_NoFinancials", F.lit(first_issue))
   else:
    financial_not_nulls = [F.col(c) > 0 for c in financial_cols]
    financial_any_not_null = reduce(or_, financial_not_nulls)
    dq2 = dq1.withColumn(
           "Check_RecordOnly_NoFinancials",
           F.when(
               ((F.col("Claim type (IND/MO/RO)").rlike("(?i)record|^ro$"))) & financial_any_not_null,
               "FAIL"
           ).otherwise("PASS")
    )   
   # Jurisdiction state must be valid
   valid_states = [
       "AL","Alabama","AK","Alaska","AZ","Arizona","AR","Arkansas","CA","California","CO","Colorado","CT","Connecticut","DE","Delaware","FL","Florida","GA","Georgia","HI","Hawaii","ID","Idaho","IL","Illinois","IN","Indiana","IA","Iowa","KS","Kansas","KY","Kentucky",
       "LA","Louisiana","ME","Maine","MD","Maryland","MA","Massachusetts","MI","Michigan","MN","Minnesota","MS","Mississippi","MO","Missouri","MT","Montana","NE","Nebraska","NV","Nevada","NH","New Hampshire","NJ","New Jersey","NM","New Mexico","NY","New York","NC","North Carolina","ND","North Dakota",
       "OH","Ohio","OK","Oklahoma","OR","Oregon","PA","Pennsylvania","RI","Rhode Island","SC","South Carolina","SD","South Dakota","TN","Tennessee","TX","Texas","UT","Utah","VT","Vermont","VA","Virginia","WA","Washington","WV","West Virginia","WI","Wisconsin","WY","Wyoming", "Dist of Colombia", "District of Colombia", "DC", "D.C."
   ]
   col_status = check_column_status(dq2, ["Benefit State / Jurisdiction State"])
   if col_status["Benefit State / Jurisdiction State"]:
       dq3 = dq2.withColumn("Check_ValidState", F.lit(col_status["Benefit State / Jurisdiction State"]))
   else:
        state_clean = F.trim(F.col("Benefit State / Jurisdiction State").cast("string"))
        dq3 = dq2.withColumn(
        "Check_ValidState",
        F.when(
           state_clean.isNull() | (~state_clean.isin(valid_states)),
           "FAIL"
        ).otherwise("PASS")
   )
   # Medical Only -> No Indemnity Paid
   required_for_dq4 = ["Claim type (IND/MO/RO)", "Net Indemnity Paid"]
   col_status = check_column_status(dq3, required_for_dq4)
   if any(col_status[c] for c in required_for_dq4):
       first_issue = next((col_status[c] for c in required_for_dq4 if col_status[c]), None)
       dq4 = dq3.withColumn("Check_MedOnly_NoIndemnity", F.lit(first_issue))
   else:
       dq4 = dq3.withColumn(
           "Check_MedOnly_NoIndemnity",
           F.when(
               ((F.col("Claim type (IND/MO/RO)").rlike("(?i)medical|^mo$"))) &
               ((F.col("Net Indemnity Paid") > 0)),
               "FAIL"
           ).otherwise("PASS")
       )
   # Medical Only -> No TTD Days or TTD Indicator = 'Yes'
   required_for_dq5 = ["Claim type (IND/MO/RO)", "TTD Indicator (if included in the WC loss run)", "LWD's due to TTD (if included in the WC loss run)"]
   col_status = check_column_status(dq4, required_for_dq5)
   if any(col_status[c] for c in required_for_dq5):
       first_issue = next((col_status[c] for c in required_for_dq5 if col_status[c]), None)
       dq5 = dq4.withColumn("Check_MedOnly_NoTTD", F.lit(first_issue))
   else:
       dq5 = dq4.withColumn(
           "Check_MedOnly_NoTTD",
           F.when(
               ((F.col("Claim type (IND/MO/RO)").rlike("(?i)medical|^mo$"))) &
               ((F.col("TTD Indicator (if included in the WC loss run)") == "Yes") | F.col("LWD's due to TTD (if included in the WC loss run)").isNotNull()),
               "FAIL"
           ).otherwise("PASS")
       )
   # Date Reported to TPA > Date of Loss
   cols_needed = ["Date of loss", "Date claim reported to TPA"]
   col_status = check_column_status(dq5, cols_needed)
   if any(col_status[c] for c in cols_needed):
       first_issue = next((col_status[c] for c in cols_needed if col_status[c]), None)
       dq6 = dq5.withColumn("Check_ReportedDateTPA_AfterDOL", F.lit(first_issue))
   else:
       dol_dt = F.to_date(F.col("Date of loss"))
       tpa_dt = F.to_date(F.col("Date claim reported to TPA"))
       dq6 = dq5.withColumn(
           "Check_ReportedDateTPA_AfterDOL",
           F.when(
               # Fail if any are null/unparseable OR not strictly later than DateOfLoss
               dol_dt.isNull() | (tpa_dt < dol_dt),
               "FAIL"
           ).otherwise("PASS")
       )
   # Date Reported > Date of Loss
   cols_needed = ["Date of loss", "Date claim reported to Client / Employer"]
   col_status = check_column_status(dq6, cols_needed)
   if any(col_status[c] for c in cols_needed):
       first_issue = next((col_status[c] for c in cols_needed if col_status[c]), None)
       dq7 = dq6.withColumn("Check_ReportedDateClient_AfterDOL", F.lit(first_issue))
   else:
       dol_dt = F.to_date(F.col("Date of loss"))
       client_dt = F.to_date(F.col("Date claim reported to Client / Employer"))
       dq7 = dq6.withColumn(
           "Check_ReportedDateClient_AfterDOL",
           F.when(
               # Fail if any are null/unparseable OR not strictly later than DateOfLoss
               dol_dt.isNull() | (client_dt < dol_dt),
               "FAIL"
           ).otherwise("PASS")
       )
   # Closed claims must have Closed Date
   status_col = "Claim status (Open/Closed)"
   date_closed_col = "Date closed (if closed claim)"
   cols_needed = [status_col, date_closed_col]
   col_status = check_column_status(dq7, cols_needed)
   if any(col_status[c] for c in cols_needed):
       first_issue = next((col_status[c] for c in cols_needed if col_status[c]), None)
       dq8 = dq7.withColumn("Check_ClosedClaims_DateClosed", F.lit(first_issue))
   else:
       closed_vals = ["closed", "c", "cl"]  # includes Closed/CLOSED, C, CL
       status_norm = F.lower(F.trim(F.col(status_col).cast("string")))
       date_closed_str = F.trim(F.col(date_closed_col).cast("string"))
       dq8 = dq7.withColumn(
           "Check_ClosedClaims_DateClosed",
           F.when(
               status_norm.isin(closed_vals) &
               (F.col(date_closed_col).isNull() | (date_closed_str == "")),
               "FAIL"
           ).otherwise("PASS")
       )
   # Final DQ Status Columns
   check_cols = [c for c in dq8.columns if c.startswith("Check_")]
   # DQ_Status: FAIL if any check FAILs, else PASS
   if check_cols:
       any_fail = reduce(or_, [(F.col(c) == "FAIL") for c in check_cols])
   else:
       any_fail = F.lit(False)
   dq_with_status = dq8.withColumn(
       "DQ_Status",
       F.when(any_fail, "FAIL").otherwise("PASS")
   )
   # DQ_Mapping_Status: UNMAPPED if any check column has "UNMAPPED FIELD"
   if check_cols:
       any_unmapped = reduce(or_, [(F.col(c) == "UNMAPPED FIELD") for c in check_cols])
   else:
       any_unmapped = F.lit(False)
   dq_final = dq_with_status.withColumn(
       "DQ_Mapping_Status",
       F.when(any_unmapped, "UNMAPPED").otherwise("MAPPED")
   )
   # Return entire dataset with DQ columns
   return dq_final

In [0]:
def dq_summary(dq_df, claim_id_col="Claim number"):
   check_cols = [c for c in dq_df.columns if c.startswith("Check_")]
   total_claims = dq_df.select(F.countDistinct(F.col(claim_id_col)).alias("total")).collect()[0]["total"]
   summaries = []
   for c in check_cols:
       per_check = (
           dq_df.select(
               F.lit(c).alias("check_name"),
               F.countDistinct(F.when(F.col(c) == "FAIL", F.col(claim_id_col))).alias("failed_claims"),
               F.countDistinct(
                   F.when(
                       (F.col(c).startswith("UNMAPPED")) | (F.col(c).startswith("DUPLICATE")),
                       F.col(claim_id_col)
                   )
               ).alias("unmapped_or_duplicate_claims")
           )
           .withColumn("total_distinct_claims", F.lit(total_claims))
           .withColumn(
               "failed_pct_of_claims",
               F.when(F.lit(total_claims) == 0, F.lit(0.0))
                .otherwise(F.col("failed_claims") / F.lit(total_claims) * 100)
           )
           .withColumn(
               "unmapped_or_duplicate_pct",
               F.when(F.lit(total_claims) == 0, F.lit(0.0))
                .otherwise(F.col("unmapped_or_duplicate_claims") / F.lit(total_claims) * 100)
           )
       )
       summaries.append(per_check)
   # union all per-check results
   if not summaries:
       return dq_df.sql_ctx.createDataFrame([], "check_name string, failed_claims long, unmapped_or_duplicate_claims long, total_distinct_claims long, failed_pct_of_claims double, unmapped_or_duplicate_pct double")
   out = summaries[0]
   for s in summaries[1:]:
       out = out.unionByName(s)
   return out.orderBy(F.desc("failed_claims"))

In [0]:
def completeness_checks(df):
    check = Check(spark, CheckLevel.Error, 'My Data Quality Checks') \
    .isUnique("Claim number") \
    .isComplete("Date of loss") \
    .isComplete("Date claim reported to TPA") \
    .isComplete("Date of birth or age (employee date of birth)") \
    .isComplete("Date of hire (employee hire date)") \
    .isComplete("Occupation") \
    .isComplete("Litigation indicator (Yes/No) ") \
    .isComplete("Claim type (IND/MO/RO)") \
    .isComplete("Claim status (Open/Closed)") \
    .isComplete("Loss description") \
    .isComplete("Cause Description") \
    .isComplete("Benefit State / Jurisdiction State") \
    .isComplete("Net Total Incurred")
    
    verification_result = VerificationSuite(spark) \
    .onData(df) \
    .addCheck(check) \
    .run()
    result_df = VerificationResult.checkResultsAsDataFrame(spark, verification_result)  

    #Filter only completeness constraints
    completeness_df = result_df.filter(F.col("constraint").contains("CompletenessConstraint"))
    #Extract column name
    completeness_df = completeness_df.withColumn(
        "column_name",
        F.regexp_extract(F.col("constraint"), r"Completeness\(([^,]+),", 1)
    )
    # Extract numeric completeness value
    completeness_df = completeness_df.withColumn(
        "completeness_value",
        F.regexp_extract(F.col("constraint_message"), r"Value:\s*([0-9]*\.?[0-9]+)", 1).cast("double")
    )
    #Compute % Failed
    completeness_df = completeness_df.withColumn(
        "% Complete",
        F.when(F.col("constraint_message").contains("Input data does not include column"), F.lit("Not Mapped"))
        .when(F.col("constraint_status") == "Success", F.format_number(F.lit(100.0), 2))
        .otherwise(F.format_number((F.col("completeness_value")) * 100, 2))
    )
    #Final clean output
    final_completeness_df = completeness_df.select(
        F.col("column_name").alias("Field Name"),
        F.col("% Complete")
    ).orderBy(F.desc("% Complete"))

    return final_completeness_df    